<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Gemini001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# HES Diffusion Diagnosis Sweep (Minimal 5D Configuration)
# GOAL: Fix the Coupling Constant (Kc) to the target Alpha and sweep the Diffusion Constant (NU)
# to find the Emergent Diffusion Constant (NU_emergent).
# This is a parameter optimization test, not a hypothesis validation test.

import numpy as np
import matplotlib.pyplot as plt

# --- SYSTEM PARAMETERS (MINIMAL 5D) ---
N = 8
GRID_SIZE = (N, N, N, N, N) # Five dimensions
D = 5 # Dimensionality

# Fixed Target Coupling Constant (Kc = Alpha)
KC_FIXED = 1 / 137.036
TOTAL_ITERATIONS = 5000
DT = 0.01

# Reaction-Diffusion Parameters (Fixed)
ALPHA_RD = 0.5
BETA = 0.01
MU = 0.005 # Mu is fixed (diffusion of mass field Phi)
BETA_LINK = 0.05

# --- INITIALIZATION ---

def initialize_fields(N, RADIUS=1, D=5):
    """Initializes all fields in a 5D grid and seeds a localized spinor particle."""
    shape = (N,) * D
    Phi = np.random.rand(*shape) * 0.1
    A = np.random.rand(*shape) * 0.1
    Psi_real = np.random.rand(*shape) * 0.1
    Psi_imag = np.random.rand(*shape) * 0.1

    CENTER = N // 2
    coords = np.indices(shape)
    squared_distance = sum((c - CENTER)**2 for c in coords)

    mask = squared_distance <= RADIUS**2
    Psi_real[mask] = 1.0
    Psi_imag[mask] = 1.0
    return Phi, A, Psi_real, Psi_imag

# --- HELPER FUNCTIONS ---

def laplacian_5d(field):
    """Calculates the 5D discrete Laplacian with periodic boundary conditions."""
    lap = -2 * D * field
    for axis in range(D):
        lap += np.roll(field, 1, axis=axis)
        lap += np.roll(field, -1, axis=axis)
    return lap

def step_simulation(Phi, A, Psi_real, Psi_imag, Kc, Nu):
    """Performs a single simulation step for all coupled fields in 5D."""

    # 1. Psi Phase/Link Dynamics
    Psi = Psi_real + 1j * Psi_imag
    Theta = np.angle(Psi)
    Magnitude = np.abs(Psi)
    mean_theta = np.mean(Theta)
    phase_correction = -BETA_LINK * (Theta - mean_theta)

    # 2. Field Dynamics (Reaction-Diffusion)

    # --- Field Phi (Mass/Energy Scalar) ---
    lap_phi = laplacian_5d(Phi)
    reaction_phi = ALPHA_RD * Phi * (1 - Phi) - Phi * (Magnitude**2)
    d_phi = (MU * lap_phi + reaction_phi) * DT
    Phi += d_phi

    # --- Field A (Force Carrier/Gauge Potential Analogue) ---
    lap_a = laplacian_5d(A)
    coupling_term_a = -Kc * Psi_imag
    d_a = (Nu * lap_a + coupling_term_a) * DT
    A += d_a

    # --- Field Psi (Spinor/Matter Field Analogue) ---
    lap_psi_real = laplacian_5d(Psi_real)
    lap_psi_imag = laplacian_5d(Psi_imag)

    # Mass/Coupling Terms
    mass_term_real = -Phi * Psi_imag
    mass_term_imag = Phi * Psi_real

    # Charge/A-Field Interaction (The "Force" Term)
    force_term_imag = A * Psi_real
    force_term_real = -A * Psi_imag

    # Full updates
    d_psi_real = (Nu * lap_psi_real + mass_term_real + force_term_real) * DT
    d_psi_imag = (Nu * lap_psi_imag + mass_term_imag + force_term_imag) * DT

    Psi_real += d_psi_real + (np.real(Psi) * phase_correction * DT)
    Psi_imag += d_psi_imag + (np.imag(Psi) * phase_correction * DT)

    # Normalization (Energy-conservation analogue)
    Psi_mag = np.sqrt(Psi_real**2 + Psi_imag**2)
    Psi_real = np.where(Psi_mag > 2.0, Psi_real * 2.0 / Psi_mag, Psi_real)
    Psi_imag = np.where(Psi_mag > 2.0, Psi_imag * 2.0 / Psi_mag, Psi_imag)

    return Phi, A, Psi_real, Psi_imag

# --- SWEEP FUNCTION ---
def run_nu_sweep(Nu_values, num_iterations, Kc):
    """Runs the stability test for a list of Nu (Diffusion) values at a fixed Kc."""
    results = {}
    print(f"Starting Diffusion Diagnosis Sweep in 5D at fixed K_c = {Kc:.5f}...")

    for i, Nu in enumerate(Nu_values):
        Phi_s, A_s, Psi_real_s, Psi_imag_s = initialize_fields(N)
        magnitude_history = []

        log_interval = num_iterations // 5
        print(f"\n--- Testing Nu = {Nu:.5f} ---")

        for t in range(num_iterations):
            Phi_s, A_s, Psi_real_s, Psi_imag_s = step_simulation(
                Phi_s, A_s, Psi_real_s, Psi_imag_s, Kc, Nu
            )
            current_mag = np.max(np.sqrt(Psi_real_s**2 + Psi_imag_s**2))
            magnitude_history.append(current_mag)

            if t % log_interval == 0:
                print(f"Iteration: {t} | Max Psi: {current_mag:.4f}")

            if current_mag < 0.001 and t > 100:
                 break

        # --- RIGOROUS STABILITY CHECK (Persistence Metric) ---
        history_len = len(magnitude_history)

        if history_len > num_iterations // 2:
            last_half_start_index = history_len // 2
            final_segment_start = int(0.9 * history_len)

            max_late_mag = np.max(magnitude_history[last_half_start_index:])
            final_mean_mag = np.mean(magnitude_history[final_segment_start:])

            # The stability threshold is tighter here, requiring 95% persistence
            if final_mean_mag > 0.1 and final_mean_mag >= 0.95 * max_late_mag:
                stability_status = "STABLE (Persistent)"
            else:
                stability_status = "UNBOUND / DECAYED"
        else:
             stability_status = "UNBOUND / COLLAPSED EARLY"
             final_mean_mag = 0.0

        results[Nu] = {
            'status': stability_status,
            'final_magnitude': final_mean_mag
        }

    return results

# --- MAIN EXECUTION: DEFINING THE SWEEP RANGE ---

NU_SWEEP_POINTS = 10
# Sweep range from 0.0001 (very low diffusion) up to 0.004 (below the prior failed 0.005)
NU_MIN = 0.0001
NU_MAX = 0.0040
Nu_sweep_values = np.linspace(NU_MIN, NU_MAX, NU_SWEEP_POINTS)

# Run the sweep
sweep_results = run_nu_sweep(Nu_sweep_values, TOTAL_ITERATIONS, KC_FIXED)

# --- ANALYSIS AND REPORTING ---

print("\n--- DIFFUSION DIAGNOSIS RESULTS (MINIMAL 5D) ---")
print(f"Fixed Coupling Constant (K_c): {KC_FIXED:.5f} (Alpha)")
print("---------------------------------------")

Nu_stable = []

for Nu, data in sweep_results.items():
    print(f"Nu = {Nu:.5f}: {data['status']} (Final Mean Mag: {data['final_magnitude']:.4f})")
    if data['status'].startswith('STABLE'):
        Nu_stable.append(Nu)

# Determine Nu_emergent
if Nu_stable:
    Nu_emergent = min(Nu_stable)
    print(f"\n[Claude-Prime: Nu_emergent Finding] The Emergent Diffusion Constant (Nu_emergent) required to stabilize Alpha is: {Nu_emergent:.5f}")
else:
    Nu_emergent = None
    print("\n[Claude-Prime: Nu_emergent Finding] No stable bound state was found in the tested diffusion range.")

print("\n---------------------------------------")
print("This test determines the missing parameter (Nu) required for HES to fit the known constant Alpha. If Nu_emergent is found, the HES model is viable, but requires parameter optimization.")



Starting Diffusion Diagnosis Sweep in 5D at fixed K_c = 0.00730...

--- Testing Nu = 0.00010 ---
Iteration: 0 | Max Psi: 1.4142
Iteration: 1000 | Max Psi: 1.3533
Iteration: 2000 | Max Psi: 1.2370
Iteration: 3000 | Max Psi: 1.2535
Iteration: 4000 | Max Psi: 1.7050

--- Testing Nu = 0.00053 ---
Iteration: 0 | Max Psi: 1.4142
Iteration: 1000 | Max Psi: 1.3036
Iteration: 2000 | Max Psi: 1.0972
Iteration: 3000 | Max Psi: 0.9023
Iteration: 4000 | Max Psi: 0.8225

--- Testing Nu = 0.00097 ---
Iteration: 0 | Max Psi: 1.4142
Iteration: 1000 | Max Psi: 1.2420
Iteration: 2000 | Max Psi: 0.9503
Iteration: 3000 | Max Psi: 0.7334
Iteration: 4000 | Max Psi: 0.5535

--- Testing Nu = 0.00140 ---
Iteration: 0 | Max Psi: 1.4142
Iteration: 1000 | Max Psi: 1.1856
Iteration: 2000 | Max Psi: 0.8402
Iteration: 3000 | Max Psi: 0.4146
Iteration: 4000 | Max Psi: 0.3661

--- Testing Nu = 0.00183 ---
Iteration: 0 | Max Psi: 1.4142
Iteration: 1000 | Max Psi: 1.1243
Iteration: 2000 | Max Psi: 0.7288
Iteration: 3000 